# <font color="#418FDE" size="6.5" uppercase>**Biclustering Validation**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply spectral co-clustering and spectral biclustering to matrix data. 
- Interpret row and column cluster labels and checkerboard structures. 
- Evaluate clustering with external, internal, consensus, and stability measures. 


## **1. Biclustering Concepts**

### **1.1. Biclustering Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_B/image_01_01.jpg?v=1784023726" width="250">



>* Groups rows and columns together
>* Finds hidden local patterns in data

>* Submatrices show shared row-column behavior
>* Coherence varies by data and context

>* Define rows, columns, and matrix values
>* Normalize data to reveal meaningful relationships



### **1.2. Spectral Co Clustering**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_B/image_01_02.jpg?v=1784023728" width="250">



>* Group related rows and columns together
>* Find strong paired patterns in matrix blocks

>* Linear algebra reveals hidden row-column patterns
>* Bipartite partitions expose communities in sparse data

>* Use normalized nonnegative matrix values
>* Inspect labels for meaningful coherent blocks



In [ ]:
#@title Python Code - Spectral Co Clustering

# This example builds a small checkerboard matrix.
# Spectral co-clustering finds linked row-column groups.
# The reordered heatmap should reveal clear blocks.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import SpectralCoclustering

# A fixed generator makes the synthetic matrix reproducible.
rng = np.random.default_rng(42)

# Three row groups connect strongly to three column groups.
base_blocks = np.array([[8, 1, 2], [1, 7, 2], [2, 1, 9]])
row_labels_true = np.repeat(np.arange(3), 10)
col_labels_true = np.repeat(np.arange(3), 8)

# Noise makes the task realistic while keeping values nonnegative.
matrix = base_blocks[row_labels_true[:, None], col_labels_true[None, :]]
noise = rng.normal(loc=0.0, scale=0.8, size=matrix.shape)
matrix = np.clip(matrix + noise, a_min=0.0, a_max=None)

# Spectral co-clustering assigns labels to rows and columns together.
model = SpectralCoclustering(n_clusters=3, random_state=42)
model.fit(matrix)

# Sorting by learned labels places related rows and columns together.
row_order = np.argsort(model.row_labels_)
col_order = np.argsort(model.column_labels_)
reordered_matrix = matrix[row_order][:, col_order]

# These short lines connect the output to the fitted model.
print("scikit-learn version:", sklearn.__version__)
print("Matrix shape:", matrix.shape)
print("Learned row cluster sizes:", np.bincount(model.row_labels_).tolist())
print("Learned column cluster sizes:", np.bincount(model.column_labels_).tolist())

# The heatmap shows the checkerboard structure after reordering.
fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(reordered_matrix, aspect="auto", cmap="viridis")
ax.set_title("Spectral co-clustering reordered matrix")
ax.set_xlabel("Columns sorted by learned cluster")
ax.set_ylabel("Rows sorted by learned cluster")
fig.colorbar(image, ax=ax, label="Association strength")
plt.show()



### **1.3. Spectral Biclustering**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_B/image_01_03.jpg?v=1784023732" width="250">



>* Clusters rows and columns into meaningful blocks
>* Uses spectral patterns to reveal checkerboards

>* Reorder matrices to reveal checkerboard patterns
>* Cluster transformed rows and columns into blocks

>* Prepare association matrices with careful normalization
>* Inspect reordered blocks for meaningful patterns



In [ ]:
#@title Python Code - Spectral Biclustering

# This example reveals checkerboard biclusters in matrix data.
# Spectral biclustering clusters rows and columns together.
# The reordered heatmap should show clearer block structure.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import SpectralBiclustering

# A fixed generator makes the synthetic matrix reproducible.
rng = np.random.default_rng(42)

# Each block has a different average association strength.
block_means = np.array([[8, 2, 5], [1, 7, 3], [4, 3, 9]])

# Repeat block means to create row and column groups.
base_matrix = np.repeat(np.repeat(block_means, 10, axis=0), 8, axis=1)
noise = rng.normal(loc=0.0, scale=0.6, size=base_matrix.shape)
data_matrix = np.clip(base_matrix + noise, a_min=0.0, a_max=None)

# Validate the matrix before fitting the model.
if data_matrix.shape != (30, 24):
    raise ValueError("Expected a 30 by 24 matrix for this lesson.")

# SpectralBiclustering searches for checkerboard row-column structure.
model = SpectralBiclustering(n_clusters=(3, 3), random_state=42)
model.fit(data_matrix)

# Sorting by learned labels places similar rows and columns together.
row_order = np.argsort(model.row_labels_)
column_order = np.argsort(model.column_labels_)
reordered_matrix = data_matrix[row_order][:, column_order]

# Print concise labels to connect the model with the heatmap.
print("scikit-learn version:", sklearn.__version__)
print("First 12 row labels:", model.row_labels_[:12].tolist())
print("First 12 column labels:", model.column_labels_[:12].tolist())

# The heatmap displays the discovered checkerboard arrangement.
fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(reordered_matrix, cmap="viridis", aspect="auto")
ax.set_title("Spectral biclustering reordered matrix")
ax.set_xlabel("Columns sorted by learned column cluster")
ax.set_ylabel("Rows sorted by learned row cluster")
fig.colorbar(image, ax=ax, label="Association strength")
plt.show()



## **2. Bicluster Outputs**

### **2.1. Row Column Labels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_B/image_02_01.jpg?v=1784023724" width="250">



>* Rows and columns get separate cluster labels
>* Labels connect algorithm output to real meaning

>* Biclusters combine row and column groups
>* Labels reveal meaningful group relationships

>* Cluster numbers are arbitrary identifiers
>* Infer meaning from data and context



In [ ]:
#@title Python Code - Row Column Labels

# This example maps biclustering labels to matrix blocks.
# Row and column labels define checkerboard intersections.
# The printed table shows interpretable bicluster means.

import numpy as np
import pandas as pd
from sklearn.cluster import SpectralCoclustering
import sklearn

# A small matrix is built with clear checkerboard structure.
base_blocks = np.array([[8, 2, 5], [3, 9, 4], [6, 1, 7]])
row_groups = np.repeat([0, 1, 2], 4)
column_groups = np.repeat([0, 1, 2], 3)

# Deterministic noise makes the example realistic but repeatable.
rng = np.random.default_rng(42)
data = base_blocks[row_groups[:, None], column_groups[None, :]]
data = data + rng.normal(0, 0.25, size=data.shape)

# Spectral co-clustering estimates row and column labels together.
model = SpectralCoclustering(n_clusters=3, random_state=42)
model.fit(data)

# Sorting by labels reveals the checkerboard arrangement.
row_order = np.argsort(model.row_labels_)
column_order = np.argsort(model.column_labels_)
sorted_data = data[row_order][:, column_order]

# Each cell summarizes one row-label and column-label intersection.
block_means = np.zeros((3, 3))
for row_label in range(3):
    for column_label in range(3):
        rows = model.row_labels_ == row_label
        columns = model.column_labels_ == column_label
        block_means[row_label, column_label] = data[rows][:, columns].mean()

# A compact table connects labels to their average matrix values.
summary = pd.DataFrame(
    np.round(block_means, 2),
    index=["row label 0", "row label 1", "row label 2"],
    columns=["col label 0", "col label 1", "col label 2"],
)

print("scikit-learn version:", sklearn.__version__)
print("First 12 row labels:", model.row_labels_[:12].tolist())
print("First 9 column labels:", model.column_labels_[:9].tolist())
print("Mean value at each row-label and column-label intersection:")
print(summary.to_string())



### **2.2. Checkerboard Patterns**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_B/image_02_02.jpg?v=1784023734" width="250">



>* Reordered clusters form meaningful rectangular blocks
>* Blocks show row-column value relationships

>* Compare block positions and value patterns
>* Reveal relationships across rows and columns

>* Clear blocks suggest meaningful biclustering structure
>* Noisy patterns still need careful interpretation



In [ ]:
#@title Python Code - Checkerboard Patterns

# This example builds a small checkerboard matrix.
# Row and column labels define rectangular biclusters.
# Reordering reveals the hidden block pattern.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import SpectralCoclustering
import sklearn

# The random generator makes the noisy example repeatable.
rng = np.random.default_rng(42)

# Each block mean describes one row-cluster and column-cluster interaction.
block_means = np.array([[8, 2, 6], [1, 7, 3], [5, 4, 9]])

# Build a matrix whose rows and columns are initially shuffled.
row_sizes = [8, 8, 8]
col_sizes = [6, 6, 6]

true_matrix = np.block([
    [np.full((row_sizes[i], col_sizes[j]), block_means[i, j])
     for j in range(3)]
    for i in range(3)
])

noisy_matrix = true_matrix + rng.normal(0, 0.7, size=true_matrix.shape)
row_shuffle = rng.permutation(noisy_matrix.shape[0])
col_shuffle = rng.permutation(noisy_matrix.shape[1])
shuffled_matrix = noisy_matrix[row_shuffle][:, col_shuffle]

# Spectral co-clustering estimates row and column cluster labels.
model = SpectralCoclustering(n_clusters=3, random_state=42)
model.fit(shuffled_matrix)

# Sorting by labels places similar rows and columns together.
row_order = np.argsort(model.row_labels_)
col_order = np.argsort(model.column_labels_)
reordered_matrix = shuffled_matrix[row_order][:, col_order]

# Validate that the reordered matrix has the expected small shape.
if reordered_matrix.shape != (24, 18):
    raise ValueError("The teaching matrix has an unexpected shape.")

print("scikit-learn version:", sklearn.__version__)
print("Estimated row cluster sizes:", np.bincount(model.row_labels_).tolist())
print("Estimated column cluster sizes:", np.bincount(model.column_labels_).tolist())

# The heatmap shows rectangular blocks after reordering.
fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(reordered_matrix, cmap="viridis", aspect="auto")

ax.set_title("Checkerboard Pattern After Biclustering")
ax.set_xlabel("Columns sorted by estimated column cluster")
ax.set_ylabel("Rows sorted by estimated row cluster")
fig.colorbar(image, ax=ax, label="Matrix value")
plt.show()



### **2.3. Consensus Scores**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_B/image_02_03.jpg?v=1784023730" width="250">



>* Measure repeated row and column grouping consistency
>* High consensus supports stable bicluster interpretations

>* High consensus means reliable bicluster membership
>* Low consensus signals cautious pattern interpretation

>* Compare similar biclusters by their stability
>* Use consensus to support reliable interpretations



In [ ]:
#@title Python Code - Consensus Scores

# This example measures consensus across repeated biclustering runs.
# Consensus means pairs stay grouped together repeatedly.
# Stable row and column labels support interpretation.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import SpectralCoclustering
import sklearn

# A fixed generator makes the synthetic matrix reproducible.
rng = np.random.default_rng(42)

# Three row groups and three column groups form checkerboard blocks.
row_groups = np.repeat(np.arange(3), 10)
col_groups = np.repeat(np.arange(3), 8)

# Each block has a different average value.
block_means = np.array([[8, 2, 5], [3, 9, 1], [6, 4, 10]])
clean_matrix = block_means[row_groups[:, None], col_groups[None, :]]

# Small noise keeps the checkerboard visible but realistic.
data_matrix = clean_matrix + rng.normal(0, 0.8, clean_matrix.shape)
data_matrix = data_matrix - data_matrix.min() + 0.1

if data_matrix.shape != (30, 24):
    raise ValueError("Expected a 30 by 24 teaching matrix.")

# Repeated runs show whether labels are stable.
row_pair_counts = np.zeros((30, 30))
col_pair_counts = np.zeros((24, 24))

for seed in range(10):
    model = SpectralCoclustering(n_clusters=3, random_state=seed)
    model.fit(data_matrix)
    row_pair_counts += model.row_labels_[:, None] == model.row_labels_[None, :]
    col_pair_counts += model.column_labels_[:, None] == model.column_labels_[None, :]

row_consensus = row_pair_counts / 10
col_consensus = col_pair_counts / 10

# Compare consensus inside true groups with consensus across groups.
row_same = row_groups[:, None] == row_groups[None, :]
col_same = col_groups[:, None] == col_groups[None, :]

row_inside = row_consensus[row_same].mean()
row_across = row_consensus[~row_same].mean()
col_inside = col_consensus[col_same].mean()
col_across = col_consensus[~col_same].mean()

print("scikit-learn version:", sklearn.__version__)
print("Mean row consensus inside groups:", round(row_inside, 2))
print("Mean row consensus across groups:", round(row_across, 2))
print("Mean column consensus inside groups:", round(col_inside, 2))
print("Mean column consensus across groups:", round(col_across, 2))

# The heatmap shows which row pairs repeatedly share labels.
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(row_consensus, vmin=0, vmax=1, cmap="viridis")

ax.set_title("Row Consensus Across Repeated Biclustering Runs")
ax.set_xlabel("Row index")
ax.set_ylabel("Row index")
fig.colorbar(image, ax=ax, label="Consensus score")
plt.show()



## **3. Validation Metrics**

### **3.1. External Validation Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_B/image_03_01.jpg?v=1784023739" width="250">



>* Compare biclusters with trusted outside knowledge
>* Check row and column meaning

>* Compare memberships, not arbitrary cluster names
>* High scores show alignment with known labels

>* External scores need careful interpretation
>* Combine metrics with stability and expert judgment



In [ ]:
#@title Python Code - External Validation Metrics

# This example validates biclustering with known labels.
# External metrics compare predictions with trusted categories.
# Higher agreement means better recovery of reference structure.

import numpy as np
import sklearn
from sklearn.cluster import SpectralCoclustering
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score

# A fixed generator makes the synthetic matrix reproducible.
rng = np.random.default_rng(42)

# Known row and column labels define the external standard.
true_row_labels = np.repeat([0, 1, 2], 8)
true_column_labels = np.repeat([0, 1, 2], 6)

# Each true row-column block receives a different average value.
block_means = np.array([[8, 2, 4], [3, 9, 1], [5, 2, 7]])
data = block_means[true_row_labels[:, None], true_column_labels[None, :]]

# Small noise makes the task realistic but still easy.
noisy_matrix = data + rng.normal(loc=0.0, scale=0.6, size=data.shape)

# Spectral co-clustering estimates row and column groups together.
model = SpectralCoclustering(n_clusters=3, random_state=42)
model.fit(noisy_matrix)

# External metrics ignore arbitrary cluster names.
row_ari = adjusted_rand_score(true_row_labels, model.row_labels_)
column_ari = adjusted_rand_score(true_column_labels, model.column_labels_)

# NMI is another agreement score between zero and one.
row_nmi = normalized_mutual_info_score(true_row_labels, model.row_labels_)
column_nmi = normalized_mutual_info_score(true_column_labels, model.column_labels_)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Row ARI versus known row labels: {row_ari:.3f}")
print(f"Column ARI versus known column labels: {column_ari:.3f}")
print(f"Row NMI versus known row labels: {row_nmi:.3f}")
print(f"Column NMI versus known column labels: {column_nmi:.3f}")



### **3.2. Internal Quality Scores**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_B/image_03_02.jpg?v=1784023741" width="250">



>* Assess biclusters without external labels
>* Seek dense, distinct, nonrandom patterns

>* Check coherence and separation within biclusters
>* Balance coverage, size, redundancy, and interpretability

>* Different scores favor different bicluster types
>* Use domain judgment, not scores alone



In [ ]:
#@title Python Code - Internal Quality Scores

# This script scores biclusters without external labels.
# Internal scores compare density, coherence, and coverage.
# Better biclusters should stand out from background noise.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import SpectralCoclustering
import sklearn

# A fixed generator makes the synthetic matrix reproducible.
rng = np.random.default_rng(42)
matrix = rng.normal(loc=0.0, scale=0.4, size=(30, 24))

# Three dense checkerboard blocks create hidden bicluster structure.
matrix[:10, :8] += 3.0
matrix[10:20, 8:16] += 2.5
matrix[20:30, 16:24] += 2.8

# Spectral co-clustering estimates row and column groups.
model = SpectralCoclustering(n_clusters=3, random_state=42)
model.fit(matrix)

# Validate the expected checkerboard label shapes.
if len(model.row_labels_) != matrix.shape[0]:
    raise ValueError("Row labels must match matrix rows.")
if len(model.column_labels_) != matrix.shape[1]:
    raise ValueError("Column labels must match matrix columns.")

# Internal scores use only matrix values and discovered labels.
block_means = []
block_stds = []
block_sizes = []
for cluster_id in range(model.n_clusters):
    row_mask = model.row_labels_ == cluster_id
    col_mask = model.column_labels_ == cluster_id
    block = matrix[row_mask][:, col_mask]
    block_means.append(float(block.mean()))
    block_stds.append(float(block.std()))
    block_sizes.append(int(block.size))

# Density rewards high values inside matched row-column blocks.
background_mean = float(matrix.mean())
density_lift = float(np.mean(block_means) - background_mean)

# Coherence rewards low variation inside the discovered blocks.
coherence_score = float(1.0 / (1.0 + np.mean(block_stds)))
coverage_score = float(sum(block_sizes) / matrix.size)

print("scikit-learn version:", sklearn.__version__)
print("Density lift over whole matrix:", round(density_lift, 3))
print("Coherence score, higher is tighter:", round(coherence_score, 3))
print("Coverage of matched bicluster cells:", round(coverage_score, 3))

# Reorder rows and columns to reveal the checkerboard structure.
row_order = np.argsort(model.row_labels_)
col_order = np.argsort(model.column_labels_)
ordered_matrix = matrix[row_order][:, col_order]

fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(ordered_matrix, aspect="auto", cmap="viridis")

ax.set_title("Reordered matrix after spectral co-clustering")
ax.set_xlabel("Columns sorted by discovered cluster")
ax.set_ylabel("Rows sorted by discovered cluster")
fig.colorbar(image, ax=ax, label="Matrix value")
plt.show()



### **3.3. Stability Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_23/Lecture_B/image_03_03.jpg?v=1784023743" width="250">



>* Check whether biclusters reappear after perturbations
>* Stable patterns inspire trust, not noise

>* Rerun biclustering after controlled data changes
>* Trust recurring row-column patterns, not exact labels

>* Stability complements scores and known labels
>* Consistent patterns support reliable interpretation



In [ ]:
#@title Python Code - Stability Selection

# This example tests biclustering stability with resampling.
# Stable labels should agree across repeated runs.
# The printed scores summarize reproducibility clearly.

import numpy as np
import sklearn
from sklearn.cluster import SpectralCoclustering
from sklearn.metrics import adjusted_rand_score

# A fixed generator makes the synthetic matrix reproducible.
rng = np.random.default_rng(42)
row_groups = np.repeat([0, 1, 2], 12)
col_groups = np.repeat([0, 1, 2], 10)

# The checkerboard means define the true bicluster structure.
block_means = np.array([[5, 1, 3], [2, 6, 1], [4, 2, 7]])
base_matrix = block_means[row_groups[:, None], col_groups[None, :]]
data_matrix = base_matrix + rng.normal(0, 0.6, size=base_matrix.shape)

# Validate the matrix shape before fitting the model.
if data_matrix.shape != (36, 30):
    raise ValueError("Expected a 36 by 30 matrix.")

# Fit one reference biclustering on the full matrix.
reference_model = SpectralCoclustering(n_clusters=3, random_state=42)
reference_model.fit(data_matrix)
reference_rows = reference_model.row_labels_
reference_cols = reference_model.column_labels_

# Refit after small noise perturbations and compare labels.
row_scores = []
col_scores = []
for run_number in range(20):
    noisy_matrix = data_matrix + rng.normal(0, 0.35, size=data_matrix.shape)
    model = SpectralCoclustering(n_clusters=3, random_state=42 + run_number)
    model.fit(noisy_matrix)
    row_scores.append(adjusted_rand_score(reference_rows, model.row_labels_))
    col_scores.append(adjusted_rand_score(reference_cols, model.column_labels_))

# Average adjusted Rand scores near one indicate stable groupings.
mean_row_score = float(np.mean(row_scores))
mean_col_score = float(np.mean(col_scores))
min_row_score = float(np.min(row_scores))
min_col_score = float(np.min(col_scores))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Mean row stability ARI: {mean_row_score:.3f}")
print(f"Mean column stability ARI: {mean_col_score:.3f}")
print(f"Lowest row and column ARI: {min_row_score:.3f}, {min_col_score:.3f}")
print("Interpretation: values close to 1 mean the checkerboard is reproducible.")



# <font color="#418FDE" size="6.5" uppercase>**Biclustering Validation**</font>


In this lecture, you learned to:
- Apply spectral co-clustering and spectral biclustering to matrix data. 
- Interpret row and column cluster labels and checkerboard structures. 
- Evaluate clustering with external, internal, consensus, and stability measures. 

In the next Module (Module 24), we will go over 'Matrix Decomposition'